In [ ]:
from rest_framework import viewsets
from rest_framework.permissions import IsAuthenticatedOrReadOnly
from .models import Book
from .serializers import BookSerializer

router = DefaultRouter()
router.register(r'books', BookViewSet)
urlpatterns = [
    path('api/', include(router.urls)),
]

class BookViewSet(viewsets.ModelViewSet):
    queryset = Book.objects.all()
    serializer_class = BookSerializer
    permission_classes = [IsAuthenticatedOrReadOnly]

In [ ]:
from rest_framework_simplejwt.views import TokenObtainPairView, TokenRefreshView

urlpatterns += [
    path('api/token/', TokenObtainPairView.as_view(), name='token_obtain_pair'),
    path('api/token/refresh/', TokenRefreshView.as_view(), name='token_refresh'),
]


In [ ]:
Built-in DRF Permission Classes:-
  1. AllowAny
  2. IsAuthenticated
  3. IsAdminUser
  4. IsAuthenticatedOrReadOnly	
  5. DjangoModelPermissions
  6. DjangoObjectPermissions

In [ ]:
Ways to Authenticate & Authorize in Python APIs
**********************************************

    from rest_framework.permissions import IsAuthenticatedOrReadOnly


    
    1.  API Key
        Client sends x-api-key in headers.
        Validate on every request.
    2.  JWT (JSON Web Token)
         Stateless and scalable
    3.  OAuth2
    4.  Session / Cookie Authentication

In [ ]:
How to Secure Your Python APIs
******************************
 1. ✅ Use HTTPS (TLS/SSL) for all API traffic.
 2. Store API secrets/tokens securely in environment variables.
✅ Add CORS protection using middleware (flask-cors, FastAPI CORSMiddleware).
✅ Implement Rate Limiting 
✅ Use Input Validation to avoid SQL injection/XSS attacks.
✅ Use Role-Based Access Control (RBAC) 

In [ ]:
Example using Function-Based View (FBV)

from django.http import JsonResponse
from django.views.decorators.csrf import csrf_exempt
import json

@csrf_exempt
def my_view(request):
    if request.method == 'GET':
        return JsonResponse({'message': 'This is a GET request'})

    elif request.method == 'POST':
        data = json.loads(request.body)
        return JsonResponse({'message': 'This is a POST request', 'data': data})

    else:
        return JsonResponse({'error': 'Method not allowed'}, status=405)


In [ ]:
category_api_project/
│
├── category_api_project/
│   ├── __init__.py
│   ├── asgi.py
│   ├── settings.py
│   ├── urls.py
│   └── wsgi.py
│
├── categories/
│   ├── __init__.py
│   ├── admin.py
│   ├── apps.py
│   ├── models.py
│   ├── serializers.py
│   ├── urls.py
│   ├── views.py
│   ├── tests/
│   │   ├── __init__.py
│   │   └── test_category_api.py
│   └── email_utils.py
│
├── manage.py


In [ ]:
1. models.py
from django.db import models

class Category(models.Model):
    name = models.CharField(max_length=100, unique=True)
    description = models.TextField(null=True, blank=True)

    def __str__(self):
        return self.name


In [ ]:
✅ 2. serializers.py

from rest_framework import serializers
from .models import Category

class CategorySerializer(serializers.ModelSerializer):
    class Meta:
        model = Category
        fields = '__all__'


In [ ]:
from django.core.mail import send_mail

def send_category_email(subject, message, to_email):
    return send_mail(
        subject,
        message,
        "admin@example.com",
        [to_email],
        fail_silently=False,
    )


In [ ]:
✅ views.py

from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status
from django.shortcuts import get_object_or_404

from .models import Category
from .serializers import CategorySerializer
from .email_utils import send_category_email


class CategoryListCreateAPIView(APIView): 

    def get(self, request):
        categories = Category.objects.all()
        serializer = CategorySerializer(categories, many=True)
        return Response(serializer.data, status=status.HTTP_200_OK)

    def post(self, request):
        serializer = CategorySerializer(data=request.data)
        if serializer.is_valid():
            serializer.save()
            return Response(serializer.data, status=status.HTTP_201_CREATED)
        return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)


class CategoryRetrieveUpdateDeleteAPIView(APIView):

    def get(self, request, pk):
        category = get_object_or_404(Category, pk=pk)
        serializer = CategorySerializer(category)
        return Response(serializer.data)

    def put(self, request, pk):
        category = get_object_or_404(Category, pk=pk)
        serializer = CategorySerializer(category, data=request.data)
        if serializer.is_valid():
            serializer.save()
            return Response(serializer.data)
        return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

    def delete(self, request, pk):
        category = get_object_or_404(Category, pk=pk)
        category.delete()
        return Response(status=status.HTTP_204_NO_CONTENT)


class CategoryEmailAPIView(APIView):

    def post(self, request):
        subject = request.data.get("subject")
        message = request.data.get("message")
        email = request.data.get("email")

        send_category_email(subject, message, email)

        return Response({"msg": "Email sent successfully"}, status=200)


In [ ]:
# uRLS
from django.urls import path
from .views import (
    CategoryListCreateAPIView,
    CategoryRetrieveUpdateDeleteAPIView,
    CategoryEmailAPIView
)

urlpatterns = [
    path('', CategoryListCreateAPIView.as_view(), name="category-list"),
    path('<int:pk>/', CategoryRetrieveUpdateDeleteAPIView.as_view(), name="category-detail"),
    path('send-email/', CategoryEmailAPIView.as_view(), name="category-email"),
]


In [ ]:
# pROJECT LEVEL URL
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('api/categories/', include('categories.urls')),
]


In [ ]:
from unittest import TestCase
from unittest.mock import patch, MagicMock
from rest_framework.test import APIRequestFactory
from rest_framework import status
from categories.views import CategoryListCreateAPIView, CategoryRetrieveUpdateDeleteAPIView

class TestCategoryAPIMock(TestCase):

    def setUp(self):
        self.factory = APIRequestFactory()
        self.view_list_create = CategoryListCreateAPIView.as_view()
        self.view_detail = CategoryRetrieveUpdateDeleteAPIView.as_view()

    @patch('categories.views.Category.objects')
    def test_get_category_list_mock(self, mock_objects):
        # Mock queryset
        mock_qs = MagicMock()
        mock_objects.all.return_value = mock_qs

        # Mock serializer data
        with patch('categories.views.CategorySerializer') as mock_serializer:
            mock_serializer.return_value.data = [{'id':1, 'name':'Books'}]

            request = self.factory.get('/api/categories/')
            response = self.view_list_create(request)

            self.assertEqual(response.status_code, status.HTTP_200_OK)
            self.assertEqual(response.data, [{'id':1, 'name':'Books'}])
            mock_objects.all.assert_called_once()
            mock_serializer.assert_called_once_with(mock_qs, many=True)

    @patch('categories.views.Category.objects')
    def test_create_category_mock(self, mock_objects):
        # Mock serializer
        with patch('categories.views.CategorySerializer') as mock_serializer:
            mock_serializer_instance = mock_serializer.return_value
            mock_serializer_instance.is_valid.return_value = True
            mock_serializer_instance.save.return_value = None
            mock_serializer_instance.data = {'id':1, 'name':'Books'}

            data = {'name':'Books', 'description':'All books'}
            request = self.factory.post('/api/categories/', data, format='json')
            response = self.view_list_create(request)

            self.assertEqual(response.status_code, status.HTTP_201_CREATED)
            self.assertEqual(response.data['name'], 'Books')
            mock_serializer.assert_called_once_with(data=data)

    @patch('categories.views.get_object_or_404')
    def test_get_category_detail_mock(self, mock_get):
        # Mock category instance
        mock_category = MagicMock()
        mock_get.return_value = mock_category

        with patch('categories.views.CategorySerializer') as mock_serializer:
            mock_serializer.return_value.data = {'id':1, 'name':'Books'}

            request = self.factory.get('/api/categories/1/')
            response = self.view_detail(request, pk=1)

            self.assertEqual(response.status_code, status.HTTP_200_OK)
            self.assertEqual(response.data, {'id':1, 'name':'Books'})
            mock_get.assert_called_once_with(patch.ANY, pk=1)
            mock_serializer.assert_called_once_with(mock_category)

    @patch('categories.views.get_object_or_404')
    def test_delete_category_mock(self, mock_get):
        # Mock category instance with delete
        mock_category = MagicMock()
        mock_get.return_value = mock_category

        request = self.factory.delete('/api/categories/1/')
        response = self.view_detail(request, pk=1)

        self.assertEqual(response.status_code, status.HTTP_204_NO_CONTENT)
        mock_category.delete.assert_called_once()
